# Exercice 1 — Introduction à l'analyse des données (Facile)

## 1) Qu'est-ce que l'analyse de données ?
L'analyse de données regroupe un ensemble de techniques qui permettent de **collecter**, **organiser**, **nettoyer**, **explorer** et **interpréter** des données afin d'en tirer des informations utiles (constats, tendances, corrélations, prédictions).

## 2) Pourquoi l'analyse des données est-elle importante dans les contextes modernes ?
Dans un monde où les données sont produites en continu (applications, capteurs, réseaux sociaux, transactions, etc.), l’analyse permet :
- de **prendre de meilleures décisions** fondées sur des preuves ;
- de **détecter des tendances** et des anomalies ;
- d’optimiser des processus (coûts, délais, qualité) ;
- de personnaliser des offres et d’améliorer l’expérience utilisateur.

## 3) Trois domaines d’application actuels
1. **Santé / médecine** : analyse d’indicateurs (symptômes, imagerie, mesures biologiques) pour améliorer le diagnostic, suivre l’évolution de maladies et évaluer des traitements.
2. **Finance / marketing** : segmentation de clients, scoring de risque, détection de fraude, analyse du comportement (campagnes, conversions).
3. **Industrie / opérations (data-driven)** : maintenance prédictive (capteurs), optimisation de la logistique et réduction des temps d’arrêt.


# Exercice 2 — Chargement et analyse initiale (Kaggle / Pandas)

## Objectif
Charger les jeux de données demandés, afficher les premières lignes et fournir une brève description.

> Note : si les fichiers sont présents localement (ex: `train.csv`, `Iris.csv`, etc.), ce notebook les charge depuis le disque. Sinon, il tente un fallback (téléchargement) quand c’est possible.


In [2]:
import os
import json
import pandas as pd

BASE_DIR = os.getcwd()

def read_csv_local_first(path_candidates, **kwargs):
    """Try to read a CSV from the first existing local candidate path.
    If none exists, return None.
    """
    for p in path_candidates:
        if p and os.path.exists(p):
            return pd.read_csv(p, **kwargs)
    return None

def head_or_msg(df, name, n=5):
    if df is None:
        print(f"[{name}] introuvable localement.")
        return
    display(df.head(n))


## 2.1 — Jeu de données : *Combien de sommeil les Américains obtiennent-ils réellement ?*

Brève description (générale) : ce dataset contient des informations liées au sommeil (heures de sommeil, distributions, sous-groupes par âge/genre, etc.).

> Chargement : cherchez un fichier CSV local dont le nom ressemble à *sleep* (ex: `sleep.csv`, `Sleep.csv`, etc.).


In [ ]:
sleep_df = read_csv_local_first(
    [
        os.path.join(BASE_DIR, 'sleep.csv'),
        os.path.join(BASE_DIR, 'Sleep.csv'),
        os.path.join(BASE_DIR, 'sleep', 'sleep.csv'),
        os.path.join(BASE_DIR, 'data', 'sleep.csv'),
        os.path.join(BASE_DIR, 'data', 'Sleep.csv'),
    ],
    sep=','
)
head_or_msg(sleep_df, 'sleep')

if sleep_df is not None:
    print('Colonnes:', list(sleep_df.columns))
    print('Forme (lignes, colonnes):', sleep_df.shape)


## 2.2 — Jeu de données : *Tendances mondiales en matière de troubles de santé mentale*

Brève description : dataset décrivant des tendances (par pays, années, indicateurs) relatives à la santé mentale (prévalence, taux, etc.).


In [ ]:
mental_df = read_csv_local_first(
    [
        os.path.join(BASE_DIR, 'mental_health.csv'),
        os.path.join(BASE_DIR, 'MentalHealth.csv'),
        os.path.join(BASE_DIR, 'mental.csv'),
        os.path.join(BASE_DIR, 'data', 'mental_health.csv'),
    ]
)
head_or_msg(mental_df, 'mental_health')

if mental_df is not None:
    print('Colonnes:', list(mental_df.columns))
    print('Forme (lignes, colonnes):', mental_df.shape)


## 2.3 — Jeu de données : *Tendances mondiales en matière de... approbations de cartes de crédit*

Brève description : dataset de transactions/demandes de cartes, avec des variables numériques (montants, âges, revenus, score) et des labels (approuvée / refusée).


In [ ]:
credit_df = read_csv_local_first(
    [
        os.path.join(BASE_DIR, 'credit_card.csv'),
        os.path.join(BASE_DIR, 'CreditCard.csv'),
        os.path.join(BASE_DIR, 'credit.csv'),
        os.path.join(BASE_DIR, 'data', 'credit_card.csv'),
    ]
)
head_or_msg(credit_df, 'credit_card')

if credit_df is not None:
    print('Colonnes:', list(credit_df.columns))
    print('Forme (lignes, colonnes):', credit_df.shape)


# Exercice 3 — Identification des types de données (qualitative vs quantitative)

## Rappel
- **Quantitative** : valeurs numériques (mesures, montants, scores, comptages).
- **Qualitative** : catégories / labels (pays, type, état, classe).


In [ ]:
def classify_columns(df):
    if df is None:
        return {}
    out = {}
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_numeric_dtype(s):
            out[col] = 'quantitative'
        else:
            out[col] = 'qualitative'
    return out

def show_classification(df, name, max_cols=30):
    cls = classify_columns(df)
    if not cls:
        print(f"[{name}] classification indisponible (dataset manquant).")
        return
    items = list(cls.items())[:max_cols]
    for col, ctype in items:
        print(f"- {col}: {ctype}")
    if len(cls) > max_cols:
        print(f"(… {len(cls)-max_cols} autres colonnes)")

show_classification(sleep_df, 'sleep')
print('---')
show_classification(mental_df, 'mental_health')
print('---')
show_classification(credit_df, 'credit_card')


## Justifications (à compléter selon les colonnes affichées)
- Si une colonne est de type numérique (float/int), elle est **quantitative**.
- Si une colonne contient des libellés/textes (pays, sexe, catégories, statuts), elle est **qualitative**.
- Attention : certaines colonnes peuvent être numériques mais codées (ex: `0/1` pour des catégories).


# Exercice 4 — Iris : colonnes qualitatives vs quantitatives

> Chargement : on cherche un CSV local de type `Iris.csv` ou `iris.csv` (Kaggle).

> Si non trouvé, ce notebook utilise sklearn (si disponible) pour recréer un DataFrame.


In [ ]:
iris_df = read_csv_local_first(
    [
        os.path.join(BASE_DIR, 'Iris.csv'),
        os.path.join(BASE_DIR, 'iris.csv'),
        os.path.join(BASE_DIR, 'data', 'Iris.csv'),
        os.path.join(BASE_DIR, 'data', 'iris.csv'),
    ]
)

if iris_df is None:
    try:
        from sklearn.datasets import load_iris
        data = load_iris(as_frame=True)
        iris_df = data.frame
        # sklearn donne la colonne target sous le nom 'target' ou équivalent
    except Exception as e:
        print('Impossible de charger Iris localement et sklearn n’est pas dispo.')
        print('Erreur:', e)

head_or_msg(iris_df, 'Iris')
if iris_df is not None:
    print('Colonnes:', list(iris_df.columns))
    print('Forme:', iris_df.shape)


In [ ]:
print('Classification Iris :')
show_classification(iris_df, 'Iris', max_cols=100)


## Description attendue (pour Iris)
Dans Iris, les colonnes de mesures (longueur/largeur de sépale et de pétale) sont **quantitatives**.
La colonne de classe (`species` / `target`) est **qualitative** (catégories : setosa, versicolor, virginica).


# Exercice 5 — Compétences d'observation : sommeil US


In [ ]:
if sleep_df is not None:
    print('Colonnes potentielles (sleep) :')
    for c in sleep_df.columns:
        print('-', c)
else:
    print('dataset sleep introuvable localement')


NameError: name 'sleep_df' is not defined

: 

## Justification (à adapter à tes colonnes)
Exemples :
- Pour une **analyse de tendances** : choisir les colonnes temporelles (année/mois) et les mesures de sommeil (heures).
- Pour une **comparaison de groupes** : choisir des colonnes catégorielles (genre, âge, état) et les variables numériques (heures de sommeil).
- Pour une **corrélation / relation** : choisir plusieurs variables quantitatives (heures) et potentiellement des indicateurs associés.


# Exercice 6 — Données structurées ou non structurées

| Source | Structuré ? | Justification |
|---|---|---|
| Rapports financiers en Excel | Oui | Excel = tables/colonnes/ligne, schéma clair. |
| Photographies sur une plateforme | Non | Images = données binaires non tabulaires ; extraction nécessaire. |
| Articles de presse sur un site web | Plutôt non structuré | Texte + mise en page HTML ; extraction/normalisation requise. |
| Données d’inventaire en base relationnelle | Oui | Schéma relationnel (tables/colonnes) + contraintes. |
| Enregistrements d’entretiens (étude de marché) | Non structuré | Audio/texte brut, nécessite transcription/traitement. |


# Exercice 7 — Transformation : non structuré → structuré


## 1) Série d’articles de blog (expériences de voyage)
**Méthode** : NLP (tokenisation, extraction d’entités : pays/lieux, catégories), puis structurer en colonnes : `lieu`, `thème`, `sentiment`, `date`, etc.


## 2) Enregistrements audio (appels support client)
**Méthode** : transcription (ASR), puis extraction de champs (motifs, demandes, sentiments) et éventuellement classification (label de ticket).


## 3) Notes manuscrites (brainstorming)
**Méthode** : OCR (reconnaissance de caractères), correction/segmentation, puis structuration (thèmes, mots-clés, priorités).


## 4) Tutoriel vidéo (cuisine)
**Méthode** : extraction des sous-titres (si présents) ou transcription audio, puis repérage des étapes (ingrédients, durée, technique) via NLP/segmentation temporelle.


# Exercice 8 — Importer `train.csv` depuis Kaggle (fichier disponible via GitHub)

> Chargement : on essaie d’abord un fichier local `train.csv`.
> Si absent, le notebook indiquera comment le récupérer depuis le repo GitHub (tu peux compléter le lien).


In [ ]:
train_df = read_csv_local_first(
    [
        os.path.join(BASE_DIR, 'train.csv'),
        os.path.join(BASE_DIR, 'data', 'train.csv'),
    ]
)
head_or_msg(train_df, 'train.csv')

if train_df is not None:
    print('Forme:', train_df.shape)


# Exercice 9 — Exporter un DataFrame vers Excel et JSON


In [ ]:
df_simple = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35],
    'score': [88.5, 91.2, 79.9]
})
df_simple

excel_path = os.path.join(BASE_DIR, 'df_simple.xlsx')
json_path = os.path.join(BASE_DIR, 'df_simple.json')

df_simple.to_excel(excel_path, index=False)
df_simple.to_json(json_path, orient='records', indent=2)

print('Export Excel ->', excel_path)
print('Export JSON  ->', json_path)


# Exercice 10 — Lecture de données JSON depuis une URL

> Remplis `JSON_URL` avec l’URL fournie dans ton énoncé.


In [ ]:
JSON_URL = 'PASTE_YOUR_JSON_URL_HERE'

if JSON_URL != 'PASTE_YOUR_JSON_URL_HERE':
    json_data = pd.read_json(JSON_URL)
    display(json_data.head())
    print('Type:', type(json_data))
    print('Colonnes:', list(json_data.columns))
else:
    print('JSON_URL non renseignée : complète l’URL dans la cellule JSON_URL.')
